# Merge suppliers and purchase orders

Load both sheets from the workbook in this folder and join them on `supplier_id`.

In [8]:
from pathlib import Path

import pandas as pd

workbook_path = Path.cwd() / "T18_Supplier_scorecard_app_AI.xlsx"

suppliers = pd.read_excel(workbook_path, sheet_name="Suppliers")
purchase_orders = pd.read_excel(workbook_path, sheet_name="Purchase Orders")

In [9]:
po_summary = purchase_orders.groupby("supplier_id", as_index=False).agg(
    po_count=("po_id", "count"),
    on_time_delivery_rate=("days_late", lambda days_late: days_late.eq(0).mean() * 100),
    total_qty_rejected=("qty_rejected", "sum"),
    total_qty_ordered=("qty_ordered", "sum"),
)
po_summary["rejection_rate"] = (
    po_summary["total_qty_rejected"] / po_summary["total_qty_ordered"]
)
po_summary = po_summary.drop(columns=["total_qty_rejected", "total_qty_ordered"])

merged = suppliers.merge(
    po_summary,
    on="supplier_id",
    how="left",
)

merged.head()

,supplier_id,supplier_name,category,city,certification,years_with_us,size,po_count,on_time_delivery_rate,rejection_rate
0,SUP001,Nova Chemicals,Packaging,Chandigarh,NaN,15,Large,10,90.000000,0.024515
1,SUP002,Nova Metals,Logistics,Kochi,ISO 9001,17,Medium,12,91.666667,0.081141
2,SUP003,Prime Metals,Packaging,Kochi,NaN,1,Large,19,84.210526,0.063822
3,SUP004,Prime Polymers,Chemicals,Kochi,ISO 9001,16,Medium,11,100.000000,0.060236
4,SUP005,Shree Polymers,Components,Pune,NaN,8,Medium,13,100.000000,0.028072


In [7]:
merged.head()

,supplier_id,supplier_name,category,city,certification,years_with_us,size,po_count,on_time_delivery_rate,rejection_rate
0,SUP001,Nova Chemicals,Packaging,Chandigarh,NaN,15,Large,10,90.000000,0.024515
1,SUP002,Nova Metals,Logistics,Kochi,ISO 9001,17,Medium,12,91.666667,0.081141
2,SUP003,Prime Metals,Packaging,Kochi,NaN,1,Large,19,84.210526,0.063822
3,SUP004,Prime Polymers,Chemicals,Kochi,ISO 9001,16,Medium,11,100.000000,0.060236
4,SUP005,Shree Polymers,Components,Pune,NaN,8,Medium,13,100.000000,0.028072


In [10]:
purchase_order_kpis = purchase_orders.groupby("supplier_id").agg(
    on_time_delivery_rate=("days_late", lambda days_late: days_late.eq(0).mean() * 100),
    total_qty_rejected=("qty_rejected", "sum"),
    total_qty_ordered=("qty_ordered", "sum"),
)

purchase_order_kpis["rejection_rate"] = (
    purchase_order_kpis["total_qty_rejected"] / purchase_order_kpis["total_qty_ordered"]
)
purchase_order_kpis = purchase_order_kpis[["on_time_delivery_rate", "rejection_rate"]].reset_index()

supplier_kpis = suppliers.merge(
    purchase_order_kpis,
    on="supplier_id",
    how="left",
)

supplier_kpis.head()

,supplier_id,supplier_name,category,city,certification,years_with_us,size,on_time_delivery_rate,rejection_rate
0,SUP001,Nova Chemicals,Packaging,Chandigarh,NaN,15,Large,90.000000,0.024515
1,SUP002,Nova Metals,Logistics,Kochi,ISO 9001,17,Medium,91.666667,0.081141
2,SUP003,Prime Metals,Packaging,Kochi,NaN,1,Large,84.210526,0.063822
3,SUP004,Prime Polymers,Chemicals,Kochi,ISO 9001,16,Medium,100.000000,0.060236
4,SUP005,Shree Polymers,Components,Pune,NaN,8,Medium,100.000000,0.028072


In [11]:
purchase_order_summary = purchase_orders.groupby("supplier_id").agg(
    on_time_delivery_rate=("days_late", lambda days_late: days_late.eq(0).mean() * 100),
    total_qty_rejected=("qty_rejected", "sum"),
    total_qty_ordered=("qty_ordered", "sum"),
    average_price_change=("price_change_vs_last_po", "mean"),
    mean_responsiveness_rating=("responsiveness_rating_1to5", "mean"),
)

purchase_order_summary["rejection_rate"] = (
    purchase_order_summary["total_qty_rejected"]
    / purchase_order_summary["total_qty_ordered"]
)
purchase_order_summary = purchase_order_summary[
    [
        "on_time_delivery_rate",
        "rejection_rate",
        "average_price_change",
        "mean_responsiveness_rating",
    ]
].reset_index()

supplier_scorecard = suppliers.merge(
    purchase_order_summary,
    on="supplier_id",
    how="left",
)

supplier_scorecard.head()

,supplier_id,supplier_name,category,city,certification,years_with_us,size,on_time_delivery_rate,rejection_rate,average_price_change,mean_responsiveness_rating
0,SUP001,Nova Chemicals,Packaging,Chandigarh,NaN,15,Large,90.000000,0.024515,0.017400,2.700000
1,SUP002,Nova Metals,Logistics,Kochi,ISO 9001,17,Medium,91.666667,0.081141,0.006250,3.000000
2,SUP003,Prime Metals,Packaging,Kochi,NaN,1,Large,84.210526,0.063822,0.025211,3.578947
3,SUP004,Prime Polymers,Chemicals,Kochi,ISO 9001,16,Medium,100.000000,0.060236,0.017091,3.454545
4,SUP005,Shree Polymers,Components,Pune,NaN,8,Medium,100.000000,0.028072,-0.010385,3.153846


In [13]:
def normalize_supplier_metrics(scorecard):
    """Normalize supplier KPIs to 0-100, keeping missing values missing."""
    normalized = scorecard.copy()

    for metric in ["on_time_delivery_rate", "mean_responsiveness_rating"]:
        values = normalized[metric]
        minimum = values.min()
        maximum = values.max()
        score_column = f"{metric}_score"
        if pd.isna(minimum) or pd.isna(maximum):
            normalized[score_column] = values
        elif minimum == maximum:
            normalized[score_column] = values.where(values.isna(), 50.0)
        else:
            normalized[score_column] = (values - minimum) / (maximum - minimum) * 100

    rejection_rate_pct = normalized["rejection_rate"] * 100
    normalized["rejection_rate_score"] = (100 - rejection_rate_pct).clip(lower=0)

    price_change = normalized["average_price_change"]
    minimum_price_change = price_change.min()
    maximum_price_change = price_change.max()
    if pd.isna(minimum_price_change) or pd.isna(maximum_price_change):
        normalized["average_price_change_score"] = price_change
    elif minimum_price_change == maximum_price_change:
        normalized["average_price_change_score"] = price_change.where(price_change.isna(), 50.0)
    else:
        normalized["average_price_change_score"] = (
            (maximum_price_change - price_change)
            / (maximum_price_change - minimum_price_change)
            * 100
        )

    return normalized

normalized_supplier_scorecard = normalize_supplier_metrics(supplier_scorecard)
normalized_supplier_scorecard.head()

,supplier_id,supplier_name,category,city,certification,years_with_us,size,on_time_delivery_rate,rejection_rate,average_price_change,mean_responsiveness_rating,on_time_delivery_rate_score,mean_responsiveness_rating_score,rejection_rate_score,average_price_change_score
0,SUP001,Nova Chemicals,Packaging,Chandigarh,NaN,15,Large,90.000000,0.024515,0.017400,2.700000,43.333333,41.759777,97.548502,45.047432
1,SUP002,Nova Metals,Logistics,Kochi,ISO 9001,17,Medium,91.666667,0.081141,0.006250,3.000000,52.777778,58.100559,91.885892,67.099964
2,SUP003,Prime Metals,Packaging,Kochi,NaN,1,Large,84.210526,0.063822,0.025211,3.578947,10.526316,89.635401,93.617802,29.599729
3,SUP004,Prime Polymers,Chemicals,Kochi,ISO 9001,16,Medium,100.000000,0.060236,0.017091,3.454545,100.000000,82.859319,93.976440,45.658753
4,SUP005,Shree Polymers,Components,Pune,NaN,8,Medium,100.000000,0.028072,-0.010385,3.153846,100.000000,66.480447,97.192772,100.000000
